# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar. Sesuaikan nama kolom dan aturan validasi dengan dataset masing-masing.

In [1]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data').exists():      
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / '*.parquet'   
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'dataset_bersih.parquet'

FILES = []
for f in sorted((PROJECT_ROOT / 'data' / 'raw').glob('*.parquet')):
    try:
        pl.scan_parquet(f).select(pl.len()).collect()
        FILES.append(f.as_posix())
    except Exception:
        print(f'File dilewati (tidak bisa dibaca): {f.name}')

if not FILES:
    raise FileNotFoundError(
        f'Dataset tidak ditemukan: {DATA_PATH}. '
        'Pastikan Milestone 1 sudah menempatkan data di data/raw/.'
    )

# news_raw_data (kolom terbesar) tidak dimuat utuh: hanya tanggal terbitnya yang diambil
POLA_PUBLISHED = r"'published':\s*'(\d{4}-\d{2}-\d{2}[T ][0-9:.+\-Zz]+)'"
lf = pl.scan_parquet(FILES)
df = (
    lf.with_columns(pl.col('news_raw_data').str.extract(POLA_PUBLISHED, 1).alias('news_raw_published'))
    .drop('news_raw_data')
    .collect(engine='streaming')
)
print(f'Ukuran awal: {df.shape[0]:,} baris x {df.shape[1]} kolom')
df.head()

Ukuran awal: 1,628,288 baris x 15 kolom


id,news_title,news_source,news_author,news_hostname,news_date,news_text,news_tags,news_kabkot,news_intent,news_image,update_date,create_date,news_guid,news_raw_published
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""1924973""","""Dana Kegiatan Isbat Nikah Baka…","""https://radarutara.disway.id/m…","""Wahyudi""","""disway.id""","""2026-04-27 01:30:00+00:00""","""Dana Kegiatan Isbat Nikah Baka…","""['apbd perubahan 2026', 'diaju…","""Kabupaten Mukomuko""","""{'intent': 'lapor_informasi', …",null,"""2026-04-27 03:08:50.074774+00:…",null,"""CBMiswFBVV95cUxOaDhvMVNUZFRlel…","""2026-04-27T08:30:00+07:00"""
"""1925951""","""Satu Orang Meninggal dari Insi…","""https://www.ayokaltim.com/satu…","""Iswanto""","""ayokaltim.com""","""2026-04-27 03:19:46+00:00""","""Satu Orang Meninggal dari Insi…","""['Peristiwa di Samarinda;Tabra…","""Kota Samarinda""","""{'intent': 'lapor_informasi', …",null,"""2026-04-27 05:26:20.891514+00:…",null,"""CBMimgFBVV95cUxQSkNndGh1VGxOek…","""2026-04-27T10:19:46+07:00"""
"""1925952""","""DPRD Bongkar Ketidaksesuaian L…","""https://infosatu.co/dprd-bongk…","""Emmy Haryanti""","""infosatu.co""","""2026-04-27 03:59:29+00:00""","""Samarinda, infosatu.co – Klaim…","""['Abdul rohim;DPRD Samarinda;L…","""Kota Samarinda""","""{'intent': 'lapor_informasi', …",null,"""2026-04-27 05:26:20.891632+00:…",null,"""CBMirwFBVV95cUxOMFMxOGNOWlZtT1…","""2026-04-27T10:59:29+07:00"""
"""1925953""","""DPRD Dorong Penanganan Banjir …","""https://infosatu.co/dprd-doron…","""Emmy Haryanti""","""infosatu.co""","""2026-04-27 01:43:24+00:00""","""Samarinda, infosatu.co – Dewan…","""['Banjir;Deni Hakim Anwar;DPRD…","""Kota Samarinda""","""{'intent': 'lapor_informasi', …",null,"""2026-04-27 05:26:20.891650+00:…",null,"""CBMirwFBVV95cUxPMGdaZU9wVTR0ZG…","""2026-04-27T08:43:24+07:00"""
"""1924974""","""Kepung Pasar Karang Nabire, Ma…","""https://papuatengah.tribunnews…","""Calvin Eluis Erari""","""tribunnews.com""","""2026-04-27 02:14:39+00:00""","""Demo di Nabire Kepung Pasar Ka…","""[]""","""Kabupaten Nabire""","""{'intent': 'lapor_informasi', …",null,"""2026-04-27 03:09:48.135779+00:…",null,"""CBMi1wFBVV95cUxNbHNySlhzMDFFV1…","""2026-04-27T09:14:39+07:00"""


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [2]:
connection = duckdb.connect()
connection.register('dataset', df)

profil = connection.execute('SUMMARIZE SELECT * FROM dataset').pl()
profil

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,i32,i32,i32,i32,i32,i64,"decimal[9,2]"
"""id""","""VARCHAR""","""1""","""999999""",1828769,null,null,null,null,null,1628288,0.00
"""news_title""","""VARCHAR""",""" - MediaPATRIOT.CO.ID""","""🪨 Produksi Batu Bara 2025 dan …",1537550,null,null,null,null,null,1628288,0.00
"""news_source""","""VARCHAR""","""http://aceh.navigasi.co.id/amp…","""https://zuper.fajar.co.id/2026…",1885675,null,null,null,null,null,1628288,0.00
"""news_author""","""VARCHAR""","""$YM""","""𝗗𝟯𝗹 𝟬𝟮""",56989,null,null,null,null,null,1628288,9.18
"""news_hostname""","""VARCHAR""","""1kabar.com""","""zuper.fajar.co.id""",5196,null,null,null,null,null,1628288,0.00
…,…,…,…,…,…,…,…,…,…,…,…
"""news_image""","""VARCHAR""","""//www.mentawaikita.com/assets/…","""uang-kertas-rupiah.jpg""",274392,null,null,null,null,null,1628288,78.77
"""update_date""","""VARCHAR""","""2025-10-07 11:27:24.717325+00:…","""2026-05-08 14:46:50.892578+00:…",1331398,null,null,null,null,null,1628288,0.00
"""create_date""","""VARCHAR""","""2025-10-07 08:59:33.262308+00:…","""2025-12-21 02:35:28.637530+00:…",362597,null,null,null,null,null,1628288,78.67


## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [3]:
jumlah_baris = len(df)
null_summary = (
    df.null_count()
    .transpose(include_header=True, header_name='kolom', column_names=['jumlah_null'])
    .with_columns(
        (pl.col('jumlah_null') / jumlah_baris * 100).round(2).alias('persen_null')
    )
    .filter(pl.col('jumlah_null') > 0)
    .sort('jumlah_null', descending=True)
)
null_summary

kolom,jumlah_null,persen_null
str,u32,f64
"""news_image""",1282636,78.77
"""create_date""",1281020,78.67
"""news_raw_published""",452632,27.8
"""news_guid""",347268,21.33
"""news_author""",149514,9.18
"""news_date""",84332,5.18


## 3. Penanganan Missing Values

Ganti `kolom_numerik` dan `kolom_kategorik` dengan nama kolom asli pada dataset. Pilih strategi sesuai jenis data: median/mean untuk numerik, nilai default atau modus untuk kategorik.

In [4]:
from datetime import datetime, timezone

# Kolom dibuang: news_image (~74% null), create_date (~74%), news_guid (~26%) -> terlalu kosong / tidak dipakai
KOLOM_DIBUANG = ['news_image', 'create_date', 'news_guid']
ZERO_WIDTH = r'[\x{200B}\x{200C}\x{200D}\x{2060}\x{FEFF}]'
PLACEHOLDER = ['', 'null', 'none', 'nan', 'n/a', 'na', '-', '--', 'unknown', 'undefined']
FORMAT_TANGGAL = ['%Y-%m-%d %H:%M:%S%.f%#z', '%Y-%m-%dT%H:%M:%S%.f%#z', '%Y-%m-%dT%H:%M:%S%.fZ',
                  '%Y-%m-%d %H:%M:%S%.f', '%Y-%m-%dT%H:%M:%S%.f']
TGL_MIN = datetime(2015, 1, 1, tzinfo=timezone.utc)
TGL_MAX = datetime.now(timezone.utc)


def parse_dt(e):
    return pl.coalesce([e.str.to_datetime(format=f, strict=False, time_zone='UTC') for f in FORMAT_TANGGAL])


def rapi(nama):  # hapus karakter tak terlihat, rapikan spasi
    return pl.col(nama).str.replace_all(ZERO_WIDTH, '').str.replace_all(r'\s+', ' ').str.strip_chars()


penulis = rapi('news_author')
update = parse_dt(pl.col('update_date'))
update_ok = pl.when(update.is_between(TGL_MIN, TGL_MAX)).then(update)


def valid(d):  # tanggal masuk akal dan tidak terbit setelah update_date
    return (d.is_between(TGL_MIN, TGL_MAX) & (update_ok.is_null() | (d <= update_ok + pl.duration(days=1)))).fill_null(False)


tgl_news, tgl_raw = parse_dt(pl.col('news_date')), parse_dt(pl.col('news_raw_published'))

df_missing_handled = df.with_columns([
    # kategorik: penulis kosong / placeholder / berisi URL -> nilai default
    pl.when(penulis.is_null() | penulis.str.to_lowercase().is_in(PLACEHOLDER) | penulis.str.contains(r'(?i)^(https?://|www\.)'))
      .then(pl.lit('Tidak diketahui')).otherwise(penulis).alias('news_author'),
    # tanggal: news_date yang kosong/tidak valid dipulihkan dari news_raw_data, selain itu dibiarkan null
    pl.when(valid(tgl_news)).then(tgl_news).when(valid(tgl_raw)).then(tgl_raw).alias('tanggal_terbit'),
    pl.when(valid(tgl_news)).then(pl.lit('news_date')).when(valid(tgl_raw)).then(pl.lit('news_raw_data'))
      .otherwise(pl.lit('tidak_tersedia')).alias('sumber_tanggal'),
    update_ok.alias('update_date'),
    # rapikan teks
    rapi('news_title').alias('news_title'),
    rapi('news_text').alias('news_text'),
    rapi('news_kabkot').alias('news_kabkot'),
    pl.col('news_hostname').str.strip_chars().str.to_lowercase().str.replace(r'^www\.', '').alias('news_hostname'),
]).with_columns(
    # kolom numerik turunan untuk analisis outlier: jumlah kata artikel
    pl.col('news_text').str.count_matches(r'\S+').cast(pl.Int64).alias('jumlah_kata')
).drop(KOLOM_DIBUANG + ['news_date', 'news_raw_published'])

print(f'Null sebelum: {df.null_count().sum_horizontal().sum():,}')
print(f'Null sesudah: {df_missing_handled.null_count().sum_horizontal().sum():,}')

Null sebelum: 3,597,402
Null sesudah: 85,604


## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

In [5]:
# Kolom numerik utama: jumlah kata artikel (dataset tidak punya kolom numerik asli)
KOLOM_NUMERIK = 'jumlah_kata'

q1 = df_missing_handled[KOLOM_NUMERIK].quantile(0.25)
q3 = df_missing_handled[KOLOM_NUMERIK].quantile(0.75)
iqr = q3 - q1
batas_bawah = q1 - 1.5 * iqr
batas_atas = q3 + 1.5 * iqr

print(f'Q1={q1}, Q3={q3}, IQR={iqr}')
print(f'Batas bawah: {batas_bawah}, Batas atas: {batas_atas}')

# Windsorization: clip nilai ke batas yang valid, bukan menghapus barisnya
df_outlier_handled = df_missing_handled.with_columns(
    pl.col(KOLOM_NUMERIK).clip(lower_bound=int(batas_bawah), upper_bound=int(batas_atas))
)

Q1=138.0, Q3=334.0, IQR=196.0
Batas bawah: -156.0, Batas atas: 628.0


## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

In [6]:
# URL artikel (id sudah unik sejak Milestone 1, jadi tidak ada duplikat berdasarkan id)
KOLOM_KUNCI = 'news_source'

total_baris = len(df_outlier_handled)
baris_unik = df_outlier_handled[KOLOM_KUNCI].n_unique() if KOLOM_KUNCI in df_outlier_handled.columns else total_baris
print(f'Total baris: {total_baris:,}')
print(f'Baris unik berdasarkan {KOLOM_KUNCI}: {baris_unik:,}')
print(f'Duplikat: {total_baris - baris_unik:,}')

df_dedup = df_outlier_handled.unique(
    subset=[KOLOM_KUNCI] if KOLOM_KUNCI in df_outlier_handled.columns else None,
    keep='first',
    maintain_order=True,
)

Total baris: 1,628,288
Baris unik berdasarkan news_source: 1,628,288
Duplikat: 0


## 6. Validasi Data Dasar

Tulis beberapa aturan validasi sederhana untuk memastikan data bersih memenuhi batas yang masuk akal. Gantikan contoh aturan di bawah dengan aturan yang relevan dengan dataset Anda.

In [7]:
connection.register('dataset_bersih', df_dedup)

validasi = connection.execute(f"""
    SELECT
        COUNT(*) AS total_baris,
        SUM(CASE WHEN {KOLOM_NUMERIK} IS NULL THEN 1 ELSE 0 END) AS pelanggaran_null,
        SUM(CASE WHEN {KOLOM_NUMERIK} < {batas_bawah} OR {KOLOM_NUMERIK} > {batas_atas}
            THEN 1 ELSE 0 END) AS pelanggaran_batas,
        COUNT(*) - COUNT(DISTINCT id) AS pelanggaran_id_duplikat,
        COUNT(*) - COUNT(DISTINCT news_source) AS pelanggaran_url_duplikat,
        SUM(CASE WHEN news_title IS NULL OR news_text IS NULL OR news_author IS NULL
                 OR news_source IS NULL THEN 1 ELSE 0 END) AS pelanggaran_kolom_wajib_null,
        SUM(CASE WHEN NOT regexp_matches(news_source, '^https?://') THEN 1 ELSE 0 END) AS pelanggaran_format_url,
        SUM(CASE WHEN tanggal_terbit < TIMESTAMPTZ '{TGL_MIN:%Y-%m-%d} 00:00:00+00'
                  OR tanggal_terbit > TIMESTAMPTZ '{TGL_MAX:%Y-%m-%d %H:%M:%S}+00'
            THEN 1 ELSE 0 END) AS pelanggaran_tanggal
    FROM dataset_bersih
""").pl()
validasi

total_baris,pelanggaran_null,pelanggaran_batas,pelanggaran_id_duplikat,pelanggaran_url_duplikat,pelanggaran_kolom_wajib_null,pelanggaran_format_url,pelanggaran_tanggal
i64,"decimal[38,0]","decimal[38,0]",i64,i64,"decimal[38,0]","decimal[38,0]","decimal[38,0]"
1628288,0,0,0,0,0,0,0


## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [8]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.write_parquet(OUTPUT_PATH)

print(f'Baris sebelum cleaning: {len(df):,}')
print(f'Baris sesudah cleaning: {len(df_dedup):,}')
print(f'Data bersih tersimpan di: {OUTPUT_PATH}')

Baris sebelum cleaning: 1,628,288
Baris sesudah cleaning: 1,628,288
Data bersih tersimpan di: D:\Semester 7\Big Data\tugas1-MukhamadSofyan\data\processed\dataset_bersih.parquet


## Catatan Milestone 2

Dokumentasikan di sini: dimensi kualitas data yang paling bermasalah, strategi yang dipilih untuk missing values dan outlier, jumlah baris yang berubah, serta aturan validasi yang diterapkan.